# Environment Setup

In [ ]:
# !pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 10.8 MB/s eta 0:00:00


# Train–Test Split

In [2]:
import pandas as pd

df = pd.read_csv('Data/cell2cell-duke univeristy.csv')
df = df.drop(columns=['Unnamed: 0', 'X', 'customer', 'churndep', 'traintest'])
df

,churn,revenue,mou,recchrge,directas,overage,roam,changem,changer,dropvce,...,retaccpt,newcelly,newcelln,refer,incmiss,income,mcycle,setprcm,setprc,retcall
0,0,57.492500,482.75,37.424999,0.2475,22.75,0.0,532.25,50.987499,8.333333,...,0,0,1,0,0,5,0,0,149.989990,0
1,0,82.275002,1312.25,75.000000,1.2375,0.00,0.0,156.75,8.145000,52.000000,...,0,1,0,0,0,6,0,0,9.989998,0
2,0,31.662500,25.50,29.990000,0.2475,0.00,0.0,59.50,4.027500,0.000000,...,0,0,1,0,0,9,0,0,29.989990,0
3,0,62.127499,97.50,65.985001,2.4750,0.00,0.0,23.50,6.822500,0.000000,...,0,1,0,0,0,6,0,0,29.989990,0
4,0,25.225000,2.50,25.000000,0.0000,0.00,0.0,-2.50,-0.225000,0.000000,...,0,1,0,0,0,7,0,0,29.989990,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71042,1,117.489998,384.00,29.990000,0.0000,250.00,0.0,0.00,0.000000,4.000000,...,0,0,0,0,0,2,0,0,29.989990,0
71043,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,...,0,0,0,0,0,6,0,1,0.000000,0
71044,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.333333,...,0,0,0,0,0,6,0,0,59.989990,0
71045,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,...,0,1,0,0,0,8,0,1,0.000000,0


In [3]:
df= df.dropna()

In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

y = df['churn'].astype(int)

df = pd.DataFrame(df)
label_encoder = LabelEncoder()


X=df.drop(['churn'], axis=1)

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Base model

In [6]:
import numpy as np
import optuna
import time
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix, classification_report, roc_auc_score)
from imblearn.over_sampling import SMOTE

overall_start_time = time.time()

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 500)

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        verbosity=0,
        use_label_encoder=False
    )

    model.fit(X_resampled, y_resampled)
    y_prob = model.predict_proba(X_test)[:, 1]

    return roc_auc_score(y_test, y_prob)


start_time = time.time()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

end_time = time.time()

total_time = end_time - start_time
print(f"Thời gian chạy tìm kiếm siêu tham số: {total_time:.2f} Giây")

best_params = study.best_params
print("Siêu tham số tối ưu:", best_params)

best_model = xgb.XGBClassifier(
    objective='binary:logistic',
    # scale_pos_weight=scale_pos_weight,
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    n_estimators=best_params['n_estimators'],
    verbosity=0,
    use_label_encoder=False
)
best_model.fit(X_resampled, y_resampled)

# Dự đoán
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Tính toán các chỉ số đánh giá
auc_score = roc_auc_score(y_test, y_prob)
print("AUC:", auc_score)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Matthews Correlation Coefficient:", matthews_corrcoef(y_test, y_pred))

print("\nMa trận nhầm lẫn:")
print(confusion_matrix(y_test, y_pred))

print("\nBáo cáo phân loại:")
print(classification_report(y_test, y_pred))

# Ghi lại thời gian kết thúc toàn bộ quá trình
overall_end_time = time.time()

# Tính tổng thời gian chạy toàn bộ
overall_total_time = overall_end_time - overall_start_time
print(f"Toàn bộ thời gian chạy: {overall_total_time:.2f} Giây")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-15 22:42:19,215] A new study created in memory with name: no-name-053b3b05-304a-4a07-81f5-2cc20abd54af
[I 2025-12-15 22:42:20,764] Trial 0 finished with value: 0.6460467060916442 and parameters: {'max_depth': 10, 'learning_rate': 0.23889054599109688, 'n_estimators': 330}. Best is trial 0 with value: 0.6460467060916442.
[I 2025-12-15 22:42:21,489] Trial 1 finished with value: 0.6624778865607378 and parameters: {'max_depth': 6, 'learning_rate': 0.13877329529788449, 'n_estimators': 396}. Best is trial 1 with value: 0.6624778865607378.
[I 2025-12-15 22:42:22,296] Trial 2 finished with value: 0.6499184887199725 and parameters: {'max_depth': 8, 'learning_rate': 0.18358067054184538, 'n_estimators': 281}. Best

Thời gian chạy tìm kiếm siêu tham số: 122.43 Giây
Siêu tham số tối ưu: {'max_depth': 5, 'learning_rate': 0.10131957052740347, 'n_estimators': 500}
AUC: 0.6712393179456736
Accuracy: 0.7130284230269802
Precision: 0.5038033937975425
Recall: 0.2157354046604861
F1 Score: 0.3021052631578947
Matthews Correlation Coefficient: 0.17880279873167124

Ma trận nhầm lẫn:
[[9023  848]
 [3130  861]]

Báo cáo phân loại:
              precision    recall  f1-score   support

           0       0.74      0.91      0.82      9871
           1       0.50      0.22      0.30      3991

    accuracy                           0.71     13862
   macro avg       0.62      0.56      0.56     13862
weighted avg       0.67      0.71      0.67     13862

Toàn bộ thời gian chạy: 123.54 Giây


## SHAP Values

In [7]:
import numpy as np
import shap
explainer = shap.Explainer(best_model)
shap_values = explainer.shap_values(X_train)


#Get shap value summary
shap_values
result = [np.mean([abs(items[i]) for items in shap_values]) for i in range(len(shap_values[0]))]

#Get column name
X.columns
result
# DataFrame
df_ = pd.DataFrame({'Feature': X.columns, 'mean|shap value|': result})
sorted_df = df_.sort_values(by='mean|shap value|',ascending=False)
# DataFrame
display(sorted_df)
sorted_df.to_csv("shap_values_sorted.csv", index=False)

,Feature,mean|shap value|
26,eqpdays,0.375417
1,mou,0.255777
21,months,0.248649
6,changem,0.183162
63,setprcm,0.175529
...,...,...
51,mailflag,0.001144
62,mcycle,0.000893
19,callfwdv,0.000607
43,occhmkr,0.000311


In [8]:
profit_table = {
    "eqpdays": 0.3922971,
    "mou": 0.27383575,
    "months": 0.2683537,
    "changem": 0.19033928,
    "phones": 0.15566355,
    "setprcm": 0.15500288,
    "overage": 0.12510847,
    "ownrent": 0.12364982,
    "creditaa": 0.111818574,
    "recchrge": 0.11178479,
    "directas": 0.10951055,
    "creditcd": 0.108519934,
    "dropvce": 0.0966647,
    "income": 0.095517606,
    "threeway": 0.09034747,
    "mailres": 0.08870609,
    "incmiss": 0.088341065,
    "prizmub": 0.081193425,
    "age1": 0.08063661,
    "blckvce": 0.07955432,
    "age2": 0.078548335,
    "uniqsubs": 0.07630953,
    "webcap": 0.075054586,
    "roam": 0.0738571,
    "models": 0.06989823,
    "peakvce": 0.06980787,
    "setprc": 0.06712751,
    "mourec": 0.066761546,
    "actvsubs": 0.064068116,
    "credita": 0.05903869,
    "opeakvce": 0.058492433,
    "revenue": 0.057051297,
    "changer": 0.052262787,
    "incalls": 0.05091391,
    "marryun": 0.050451566,
    "marryyes": 0.04999858,
    "callwait": 0.045237325,
    "newcelly": 0.044960454,
    "custcare": 0.04222929,
    "outcalls": 0.041492436,
    "retcalls": 0.03418307,
    "unansvce": 0.032022763,
    "prizmtwn": 0.030589614,
    "occprof": 0.030486966,
    "refurb": 0.027565725,
    "dropblk": 0.026701743,
    "truck": 0.022587338,
    "newcelln": 0.020809047,
    "prizmrur": 0.015735166,
    "refer": 0.013898367,
    "mailord": 0.013114,
    "pcown": 0.012731194,
    "occcrft": 0.010717165,
    "children": 0.009005423,
    "travel": 0.006408993,
    "occself": 0.0037108439,
    "rv": 0.0026592917,
    "occcler": 0.0024751162,
    "occret": 0.0019850743,
    "occstud": 0.0019252233,
    "retaccpt": 0.0018356557,
    "mailflag": 0.0014303686,
    "callfwdv": 0.00080224825,
    "mcycle": 0.0007587086,
    "occhmkr": 0.00045569523,
    "retcall": 0.0
}


# HAFCP Model For Training Dataset

## Highly Associated Fuzzy Churn Patterns in Binary Classification

### Fuzzy Triangle & Gaussion

In [ ]:
X_temp=pd.concat([X_train,y_train],axis=1)

# Lấy ra các row nhãn là churn
X_churn_label = X_temp[X_temp['churn'] == 1]

X_churn_label

,revenue,mou,recchrge,directas,overage,roam,changem,changer,dropvce,blckvce,...,newcelly,newcelln,refer,incmiss,income,mcycle,setprcm,setprc,retcall,churn
52253,26.040001,44.25,24.990000,0.0000,0.00,0.0000,-31.25,-1.050000,0.000000,0.000000,...,1,0,0,0,5,0,1,0.00000,0,1
63241,48.867500,295.25,39.990002,0.4950,51.75,0.0000,-98.25,-17.887501,4.000000,0.000000,...,0,0,1,0,6,0,1,0.00000,0,1
67661,39.237499,37.75,44.990002,0.2475,0.00,0.0000,-4.75,-0.247500,0.333333,4.333333,...,0,0,0,0,9,0,1,0.00000,0,1
53577,111.422501,1220.00,59.990002,2.7225,171.00,4.1950,31.00,-29.882500,19.666666,1.666667,...,0,0,0,0,6,0,0,29.98999,0,1
52347,30.272499,78.25,30.000000,0.2475,0.00,0.0000,-60.25,-0.272500,0.333333,3.333333,...,0,0,0,0,4,1,0,29.98999,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61120,97.562500,832.50,49.990002,0.2475,90.50,1.5725,484.50,46.817501,3.333333,0.000000,...,1,0,0,0,8,0,0,149.98999,0,1
66341,84.989998,371.25,94.989998,0.0000,0.00,0.0000,-105.25,0.000000,0.000000,2.000000,...,0,0,0,1,0,0,0,79.98999,0,1
68840,10.097500,75.75,10.000000,0.0000,0.25,0.0000,-42.75,-0.097500,0.000000,0.333333,...,0,0,0,0,4,0,1,0.00000,0,1
61658,99.055000,837.25,59.990002,3.2175,92.75,0.0000,-354.25,-49.064999,2.666667,14.666667,...,0,1,0,0,9,0,0,199.98999,0,1


In [ ]:
def print_column_names(X_churn_label):
    print("Các cột trong DataFrame là:")
    print(list(X_churn_label.columns))
print_column_names(X_churn_label)

Các cột trong DataFrame là:
['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall', 'churn']


In [ ]:
X_churn_label.shape

(15993, 67)

In [ ]:
Xtofuzzy_Tri  = X_churn_label[['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall']]

In [ ]:
import pandas as pd

# Membership function
def triangular_mf(x, a, b, c):
    if x <= a:
        return 0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x <= c:
        return (c - x) / (c - b)
    else:
        return 0

# Maximal cardinality
def fuzzy_classification(x, min_val, median_val, max_val):
    low_membership = triangular_mf(x, min_val, min_val, median_val)
    medium_membership = triangular_mf(x, min_val, median_val, max_val)
    high_membership = triangular_mf(x, median_val, max_val, max_val)

    max_membership = max(low_membership, medium_membership, high_membership)

    if max_membership == low_membership:
        return 0
    elif max_membership == medium_membership:
        return 1
    else:
        return 2


fuzzy_df = pd.DataFrame()
# For each feature
for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    fuzzy_df[f'{feature}_fuzzy'] = Xtofuzzy_Tri[feature].apply(lambda x: fuzzy_classification(x, min_val, median_val, max_val))

fuzzy_df

# Define a dict
boundaries = {}

for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    # Save
    boundaries[feature] = (min_val, median_val, max_val)

# print out
for feature, bounds in boundaries.items():
    print(f"Feature: {feature}")
    print(f"Low boundary: {bounds[0]}, Medium boundary: {bounds[1]}, High boundary: {bounds[2]}")
    print("------")

fuzzy_df

Feature: revenue
Low boundary: 3.75, Medium boundary: 47.62749863, High boundary: 861.1049805
------
Feature: mou
Low boundary: 0.0, Medium boundary: 330.0, High boundary: 5409.75
------
Feature: recchrge
Low boundary: -6.050000191, Medium boundary: 44.26250076, High boundary: 337.9750061
------
Feature: directas
Low boundary: 0.0, Medium boundary: 0.247500002, High boundary: 46.52999878
------
Feature: overage
Low boundary: 0.0, Medium boundary: 3.75, High boundary: 2018.0
------
Feature: roam
Low boundary: 0.0, Medium boundary: 0.0, High boundary: 850.8624878
------
Feature: changem
Low boundary: -2867.5, Medium boundary: -10.25, High boundary: 5192.25
------
Feature: changer
Low boundary: -851.1049805, Medium boundary: -0.310000002, High boundary: 2483.482422
------
Feature: dropvce
Low boundary: 0.0, Medium boundary: 3.0, High boundary: 208.6666718
------
Feature: blckvce
Low boundary: 0.0, Medium boundary: 1.0, High boundary: 314.6666565
------
Feature: unansvce
Low boundary: 0.0,

,revenue_fuzzy,mou_fuzzy,recchrge_fuzzy,directas_fuzzy,overage_fuzzy,roam_fuzzy,changem_fuzzy,changer_fuzzy,dropvce_fuzzy,blckvce_fuzzy,...,retaccpt_fuzzy,newcelly_fuzzy,newcelln_fuzzy,refer_fuzzy,incmiss_fuzzy,income_fuzzy,mcycle_fuzzy,setprcm_fuzzy,setprc_fuzzy,retcall_fuzzy
52253,1,0,1,0,0,0,1,1,0,0,...,0,2,0,0,0,1,0,1,0,0
63241,1,1,1,1,1,0,1,1,1,0,...,0,0,0,1,0,1,0,1,0,0
67661,1,0,1,1,0,0,1,1,0,1,...,0,0,0,0,0,2,0,1,0,0
53577,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,0,0,1,0
52347,1,0,1,1,0,0,1,1,0,1,...,0,0,0,0,0,1,2,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61120,1,1,1,1,1,1,1,1,1,0,...,0,2,0,0,0,2,0,0,1,0
66341,1,1,1,0,0,0,1,1,0,1,...,0,0,0,0,2,0,0,0,1,0
68840,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
61658,1,1,1,1,1,0,1,1,1,1,...,0,0,2,0,0,2,0,0,1,0


In [ ]:

df_encoded = fuzzy_df
fuzzy_columns = [col for col in df_encoded.columns]

# one-hot encoding
df_encoded = pd.get_dummies(df_encoded, columns=fuzzy_columns)

for col in fuzzy_columns:
    mappings = {
        f"{col}_{i}": f"{col}_{i}" for i in range(3)
    }
    df_encoded.rename(columns=mappings, inplace=True)
df_encoded.columns

Index(['revenue_fuzzy_0', 'revenue_fuzzy_1', 'revenue_fuzzy_2', 'mou_fuzzy_0',
       'mou_fuzzy_1', 'mou_fuzzy_2', 'recchrge_fuzzy_0', 'recchrge_fuzzy_1',
       'recchrge_fuzzy_2', 'directas_fuzzy_0',
       ...
       'income_fuzzy_2', 'mcycle_fuzzy_0', 'mcycle_fuzzy_2', 'setprcm_fuzzy_0',
       'setprcm_fuzzy_1', 'setprc_fuzzy_0', 'setprc_fuzzy_1', 'setprc_fuzzy_2',
       'retcall_fuzzy_0', 'retcall_fuzzy_2'],
      dtype='object', length=166)

In [ ]:
dataset = {}

transaction_id = 1
for _, row in df_encoded.iterrows():
    transaction = {}
    for column, value in row.items():
        transaction[column] = value
    transaction_key = f'transaction{transaction_id}'
    dataset[transaction_key] = transaction
    transaction_id += 1

In [ ]:
transactions = dataset

filtered_transactions = {transaction: {key: value for key, value in items.items() if value != 0}
                        for transaction, items in transactions.items()}

filtered_transactions

{'transaction1': {'revenue_fuzzy_1': True,
  'mou_fuzzy_0': True,
  'recchrge_fuzzy_1': True,
  'directas_fuzzy_0': True,
  'overage_fuzzy_0': True,
  'roam_fuzzy_0': True,
  'changem_fuzzy_1': True,
  'changer_fuzzy_1': True,
  'dropvce_fuzzy_0': True,
  'blckvce_fuzzy_0': True,
  'unansvce_fuzzy_0': True,
  'custcare_fuzzy_0': True,
  'threeway_fuzzy_0': True,
  'mourec_fuzzy_0': True,
  'outcalls_fuzzy_1': True,
  'incalls_fuzzy_1': True,
  'peakvce_fuzzy_0': True,
  'opeakvce_fuzzy_0': True,
  'dropblk_fuzzy_0': True,
  'callfwdv_fuzzy_0': True,
  'callwait_fuzzy_0': True,
  'months_fuzzy_1': True,
  'uniqsubs_fuzzy_0': True,
  'actvsubs_fuzzy_1': True,
  'phones_fuzzy_0': True,
  'models_fuzzy_0': True,
  'eqpdays_fuzzy_1': True,
  'age1_fuzzy_1': True,
  'age2_fuzzy_1': True,
  'children_fuzzy_2': True,
  'credita_fuzzy_0': True,
  'creditaa_fuzzy_0': True,
  'prizmrur_fuzzy_0': True,
  'prizmub_fuzzy_0': True,
  'prizmtwn_fuzzy_0': True,
  'refurb_fuzzy_0': True,
  'webcap_fuzzy

### High Utility fuzzy Churn Patterns

In [ ]:
import time

def extract_base_name(fuzzy_name):
    if fuzzy_name.startswith("Status_"):
        return "Status"
    elif fuzzy_name.startswith("Age Group_"):
        return "Age Group"
    elif fuzzy_name.startswith("Tariff Plan_"):
        return "Tariff Plan"
    elif "fuzzy" in fuzzy_name:
        return fuzzy_name.split("_", 1)[0]
    else:
        return fuzzy_name

def calculate_utility(itemset, transactions, profit_table):
    total_utility = 0
    for transaction in transactions.values():
        if set(itemset).issubset(set(transaction.keys())):
            for item, utility in transaction.items():
                if item in itemset:
                    base_name = extract_base_name(item)
                    total_utility += utility * profit_table.get(base_name, 0)
    return total_utility

def find_top_k_high_utility_itemsets(transactions, profit_table, k):
    itemsets = set()
    for transaction in transactions.values():
        itemsets.update(transaction.keys())

    high_utility_itemsets = {}
    for item in itemsets:
        utility = calculate_utility([item], transactions, profit_table)
        if utility > 0:
            high_utility_itemsets[(item,)] = utility

    P_itemsets = {}
    while True:
        temp_itemsets = {}
        for itemset, utility in high_utility_itemsets.items():
            for item in itemsets:
                if item not in itemset:
                    new_itemset = tuple(sorted(list(itemset) + [item]))
                    new_utility = calculate_utility(new_itemset, transactions, profit_table)
                    if new_utility > 0:
                        temp_itemsets[new_itemset] = new_utility
                        P_itemsets[new_itemset] = new_utility

        if not temp_itemsets:
            break

        sorted_itemsets = sorted(temp_itemsets.items(), key=lambda x: x[1], reverse=True)
        top_k_itemsets = sorted_itemsets[:k]
        high_utility_itemsets = {itemset: utility for itemset, utility in top_k_itemsets}

    return high_utility_itemsets, P_itemsets

def find_top_k_high_utility_itemsets(transactions, profit_table, k, desired_length):
    """
    Tìm ra K tập mục có độ hữu ích cao nhất với độ dài tối đa được chỉ định.
    """
    itemsets = set()
    for transaction in transactions.values():
        # Thêm từng mục trong giao dịch vào tập hợp itemsets
        itemsets.update(transaction.keys())

    # Lưu trữ các tập mục có độ hữu ích cao và tổng giá trị của chúng
    high_utility_itemsets = {}
    P_itemsets = {}

    for item in itemsets:
        utility = calculate_utility([item], transactions, profit_table)
        if utility > 0:
            high_utility_itemsets[(item,)] = utility

    while True:
        temp_itemsets = {}
        for itemset, utility in high_utility_itemsets.items():
            for item in itemsets:
                if item not in itemset:
                    # Tạo tập mục mới bằng cách kết hợp
                    new_itemset = tuple(sorted(list(itemset) + [item]))
                    new_utility = calculate_utility(new_itemset, transactions, profit_table)

                    # Kiểm tra điều kiện độ dài và độ hữu ích
                    if new_utility > 0 and len(new_itemset) <= desired_length:
                        temp_itemsets[new_itemset] = new_utility
                        P_itemsets[new_itemset] = new_utility

        if not temp_itemsets:
            break

        # Lấy top K tập mục có độ hữu ích cao nhất
        sorted_itemsets = sorted(temp_itemsets.items(), key=lambda x: x[1], reverse=True)
        top_k_itemsets = sorted_itemsets[:k]
        high_utility_itemsets = {itemset: utility for itemset, utility in top_k_itemsets}

    return high_utility_itemsets, P_itemsets



In [ ]:
result,P_itemsets= find_top_k_high_utility_itemsets(filtered_transactions, profit_table, 3,2)


sorted_data = sorted(P_itemsets.items(), key=lambda x: x[1], reverse=True)


top_10 = sorted_data[:10]
top_10 = {itemset: value for itemset, value in top_10}

print("Top 10 High Utility Itemsets:")
for itemset, utility in top_10.items():
    print(itemset, "-> Utility:", round(utility,0))

Top 10 High Utility Itemsets:
('changem_fuzzy_1', 'eqpdays_fuzzy_1') -> Utility: 7573.0
('eqpdays_fuzzy_1', 'months_fuzzy_1') -> Utility: 6384.0
('creditaa_fuzzy_0', 'eqpdays_fuzzy_1') -> Utility: 5974.0
('eqpdays_fuzzy_1', 'mou_fuzzy_1') -> Utility: 5949.0
('actvsubs_fuzzy_1', 'eqpdays_fuzzy_1') -> Utility: 5938.0
('eqpdays_fuzzy_1', 'recchrge_fuzzy_1') -> Utility: 5858.0
('changer_fuzzy_1', 'eqpdays_fuzzy_1') -> Utility: 5787.0
('changem_fuzzy_1', 'months_fuzzy_1') -> Utility: 5374.0
('credita_fuzzy_0', 'eqpdays_fuzzy_1') -> Utility: 5331.0
('eqpdays_fuzzy_1', 'webcap_fuzzy_1') -> Utility: 5324.0


In [ ]:

fuzzy_df = pd.DataFrame()
for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    fuzzy_df[f'{feature}_fuzzy'] = Xtofuzzy_Tri[feature].apply(lambda x: fuzzy_classification(x, min_val, median_val, max_val))


df_fuzzy = fuzzy_df

fuzzy_columns = [col for col in df_fuzzy.columns]

df_fuzzy = pd.get_dummies(df_fuzzy, columns=fuzzy_columns)
df_fuzzy =df_fuzzy.astype(int)
df_fuzzy

,revenue_fuzzy_0,revenue_fuzzy_1,revenue_fuzzy_2,mou_fuzzy_0,mou_fuzzy_1,mou_fuzzy_2,recchrge_fuzzy_0,recchrge_fuzzy_1,recchrge_fuzzy_2,directas_fuzzy_0,...,income_fuzzy_2,mcycle_fuzzy_0,mcycle_fuzzy_2,setprcm_fuzzy_0,setprcm_fuzzy_1,setprc_fuzzy_0,setprc_fuzzy_1,setprc_fuzzy_2,retcall_fuzzy_0,retcall_fuzzy_2
52253,0,1,0,1,0,0,0,1,0,1,...,0,1,0,0,1,1,0,0,1,0
63241,0,1,0,0,1,0,0,1,0,0,...,0,1,0,0,1,1,0,0,1,0
67661,0,1,0,1,0,0,0,1,0,0,...,1,1,0,0,1,1,0,0,1,0
53577,0,1,0,0,1,0,0,1,0,0,...,0,1,0,1,0,0,1,0,1,0
52347,0,1,0,1,0,0,0,1,0,0,...,0,0,1,1,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61120,0,1,0,0,1,0,0,1,0,0,...,1,1,0,1,0,0,1,0,1,0
66341,0,1,0,0,1,0,0,1,0,1,...,0,1,0,1,0,0,1,0,1,0
68840,1,0,0,1,0,0,1,0,0,1,...,0,1,0,0,1,1,0,0,1,0
61658,0,1,0,0,1,0,0,1,0,0,...,1,1,0,1,0,0,1,0,1,0


In [ ]:
Xy_merge = pd.merge(X_temp, df_fuzzy, left_index=True, right_index=True, how='left')
Xy_merge['fuzzyp1'] = Xy_merge['changem_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp2'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['months_fuzzy_1']
Xy_merge['fuzzyp3'] = Xy_merge['creditaa_fuzzy_0'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp4'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['mou_fuzzy_1']
Xy_merge['fuzzyp5'] = Xy_merge['actvsubs_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp6'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['recchrge_fuzzy_1']
Xy_merge['fuzzyp7'] = Xy_merge['changer_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp8'] = Xy_merge['changem_fuzzy_1'] * Xy_merge['months_fuzzy_1']
Xy_merge['fuzzyp9'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['webcap_fuzzy_1']
Xy_merge['fuzzyp10'] = Xy_merge['credita_fuzzy_0'] * Xy_merge['eqpdays_fuzzy_1']

print(Xy_merge.columns.tolist())


['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall', 'churn', 'revenue_fuzzy_0', 'revenue_fuzzy_1', 'revenue_fuzzy_2', 'mou_fuzzy_0', 'mou_fuzzy_1', 'mou_fuzzy_2', 'recchrge_fuzzy_0', 'recchrge_fuzzy_1', 'recchrge_fuzzy_2', 'directas_fuzzy_0', 'directas_fuzzy_1', 'directas_fuzzy_2', 'overage_fuzzy_0', 'overage_fuzzy_1', 'overage_fuzzy_2'

In [ ]:
Xy_merge = Xy_merge.fillna(0)

In [ ]:
columns_to_keep = [
    'revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer',
    'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls',
    'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs',
    'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur',
    'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft',
    'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord',
    'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly',
    'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall',
    'fuzzyp1', 'fuzzyp2', 'fuzzyp3', 'fuzzyp4', 'fuzzyp5', 'fuzzyp6', 'fuzzyp7', 'fuzzyp8',
    'fuzzyp9', 'fuzzyp10'
]
X_train_2 = Xy_merge[columns_to_keep]
y_train_2 = Xy_merge['churn']


# HAFCP Model For Test dataset

## Highly Associated Fuzzy Churn Patterns in Binary Classification

### Fuzzy Triangle & Gaussion

In [23]:
X_temp_test = pd.concat([X_test,y_test],axis=1)

# Lấy ra các row nhãn là churn
X_churn_label = X_temp_test[X_temp_test['churn'] == 1]

X_churn_label

,revenue,mou,recchrge,directas,overage,roam,changem,changer,dropvce,blckvce,...,newcelly,newcelln,refer,incmiss,income,mcycle,setprcm,setprc,retcall,churn
54569,34.145000,318.75,24.990000,0.000,0.00,9.1550,220.25,22.844999,0.333333,1.333333,...,0,1,0,0,6,1,1,0.00000,0,1
52439,79.964996,354.50,69.989998,0.000,54.00,0.0000,-107.50,-16.725000,19.666666,1.666667,...,0,1,0,0,7,0,0,29.98999,0,1
63639,30.260000,88.75,39.990002,0.000,0.00,0.0000,-2.75,-0.270000,0.666667,0.000000,...,0,0,0,0,3,0,1,0.00000,0,1
57383,155.212494,1342.75,47.572498,2.475,298.00,4.5325,-222.75,-37.222500,8.666667,14.000000,...,0,0,0,0,5,0,0,79.98999,0,1
70085,189.570007,1658.00,78.029999,0.000,325.75,8.6750,160.00,-9.190000,13.333333,12.333333,...,1,0,0,0,4,0,1,0.00000,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55990,31.795000,239.75,29.990000,0.000,3.25,0.4800,-83.75,-1.805000,0.333333,6.666667,...,0,0,0,0,4,0,1,0.00000,0,1
53067,32.490002,136.75,32.490002,0.000,0.00,0.0000,-1.75,0.000000,0.333333,1.666667,...,0,1,0,0,8,0,1,0.00000,0,1
53682,54.482498,411.75,59.990002,0.000,0.00,1.2175,-120.75,-1.242500,44.333332,10.666667,...,1,0,0,0,4,0,0,149.98999,0,1
51952,29.990000,253.25,39.990002,0.000,0.00,0.0000,124.75,0.000000,2.333333,1.000000,...,1,0,0,0,8,0,0,29.98999,0,1


In [24]:
def print_column_names(X_churn_label):
    print("Các cột trong DataFrame là:")
    print(list(X_churn_label.columns))
print_column_names(X_churn_label)

Các cột trong DataFrame là:
['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall', 'churn']


In [25]:
X_churn_label.shape

(3991, 67)

In [26]:
Xtofuzzy_Tri  = X_churn_label[['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall']]

In [27]:
import pandas as pd

# Membership function
def triangular_mf(x, a, b, c):
    if x <= a:
        return 0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x <= c:
        return (c - x) / (c - b)
    else:
        return 0

# Maximal cardinality
def fuzzy_classification(x, min_val, median_val, max_val):
    low_membership = triangular_mf(x, min_val, min_val, median_val)
    medium_membership = triangular_mf(x, min_val, median_val, max_val)
    high_membership = triangular_mf(x, median_val, max_val, max_val)

    max_membership = max(low_membership, medium_membership, high_membership)

    if max_membership == low_membership:
        return 0
    elif max_membership == medium_membership:
        return 1
    else:
        return 2


fuzzy_df = pd.DataFrame()
# For each feature
for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    fuzzy_df[f'{feature}_fuzzy'] = Xtofuzzy_Tri[feature].apply(lambda x: fuzzy_classification(x, min_val, median_val, max_val))

fuzzy_df

# Define a dict
boundaries = {}

for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    # Save
    boundaries[feature] = (min_val, median_val, max_val)

# print out
for feature, bounds in boundaries.items():
    print(f"Feature: {feature}")
    print(f"Low boundary: {bounds[0]}, Medium boundary: {bounds[1]}, High boundary: {bounds[2]}")
    print("------")

fuzzy_df

Feature: revenue
Low boundary: 2.537499905, Medium boundary: 47.88000107, High boundary: 595.4324951
------
Feature: mou
Low boundary: 0.0, Medium boundary: 328.25, High boundary: 4258.25
------
Feature: recchrge
Low boundary: 0.0, Medium boundary: 44.98749924, High boundary: 243.7400055
------
Feature: directas
Low boundary: 0.0, Medium boundary: 0.247500002, High boundary: 31.18499947
------
Feature: overage
Low boundary: 0.0, Medium boundary: 3.0, High boundary: 1149.25
------
Feature: roam
Low boundary: 0.0, Medium boundary: 0.0, High boundary: 151.3200073
------
Feature: changem
Low boundary: -1941.25, Medium boundary: -9.5, High boundary: 3046.75
------
Feature: changer
Low boundary: -439.5025024, Medium boundary: -0.292499989, High boundary: 619.6799927
------
Feature: dropvce
Low boundary: 0.0, Medium boundary: 3.0, High boundary: 145.3333282
------
Feature: blckvce
Low boundary: 0.0, Medium boundary: 1.0, High boundary: 286.3333435
------
Feature: unansvce
Low boundary: 0.0, M

,revenue_fuzzy,mou_fuzzy,recchrge_fuzzy,directas_fuzzy,overage_fuzzy,roam_fuzzy,changem_fuzzy,changer_fuzzy,dropvce_fuzzy,blckvce_fuzzy,...,retaccpt_fuzzy,newcelly_fuzzy,newcelln_fuzzy,refer_fuzzy,incmiss_fuzzy,income_fuzzy,mcycle_fuzzy,setprcm_fuzzy,setprc_fuzzy,retcall_fuzzy
54569,1,1,1,0,0,1,1,1,0,1,...,0,0,2,0,0,1,2,1,0,0
52439,1,1,1,0,1,0,1,1,1,1,...,0,0,2,0,0,1,0,0,1,0
63639,1,0,1,0,0,0,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
57383,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,0,0,1,0
70085,1,1,1,0,1,1,1,1,1,1,...,0,2,0,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55990,1,1,1,0,1,1,1,1,0,1,...,0,0,0,0,0,1,0,1,0,0
53067,1,0,1,0,0,0,1,1,0,1,...,0,0,2,0,0,2,0,1,0,0
53682,1,1,1,0,0,1,1,1,1,1,...,0,2,0,0,0,1,0,0,1,0
51952,1,1,1,0,0,0,1,1,1,1,...,0,2,0,0,0,2,0,0,1,0


In [28]:

df_encoded = fuzzy_df
fuzzy_columns = [col for col in df_encoded.columns]

# one-hot encoding
df_encoded = pd.get_dummies(df_encoded, columns=fuzzy_columns)

for col in fuzzy_columns:
    mappings = {
        f"{col}_{i}": f"{col}_{i}" for i in range(3)
    }
    df_encoded.rename(columns=mappings, inplace=True)
df_encoded.columns

Index(['revenue_fuzzy_0', 'revenue_fuzzy_1', 'revenue_fuzzy_2', 'mou_fuzzy_0',
       'mou_fuzzy_1', 'mou_fuzzy_2', 'recchrge_fuzzy_0', 'recchrge_fuzzy_1',
       'recchrge_fuzzy_2', 'directas_fuzzy_0',
       ...
       'income_fuzzy_2', 'mcycle_fuzzy_0', 'mcycle_fuzzy_2', 'setprcm_fuzzy_0',
       'setprcm_fuzzy_1', 'setprc_fuzzy_0', 'setprc_fuzzy_1', 'setprc_fuzzy_2',
       'retcall_fuzzy_0', 'retcall_fuzzy_2'],
      dtype='object', length=166)

In [29]:
dataset = {}

transaction_id = 1
for _, row in df_encoded.iterrows():
    transaction = {}
    for column, value in row.items():
        transaction[column] = value
    transaction_key = f'transaction{transaction_id}'
    dataset[transaction_key] = transaction
    transaction_id += 1

In [30]:
transactions = dataset

filtered_transactions = {transaction: {key: value for key, value in items.items() if value != 0}
                        for transaction, items in transactions.items()}

filtered_transactions

{'transaction1': {'revenue_fuzzy_1': True,
  'mou_fuzzy_1': True,
  'recchrge_fuzzy_1': True,
  'directas_fuzzy_0': True,
  'overage_fuzzy_0': True,
  'roam_fuzzy_1': True,
  'changem_fuzzy_1': True,
  'changer_fuzzy_1': True,
  'dropvce_fuzzy_0': True,
  'blckvce_fuzzy_1': True,
  'unansvce_fuzzy_1': True,
  'custcare_fuzzy_0': True,
  'threeway_fuzzy_1': True,
  'mourec_fuzzy_0': True,
  'outcalls_fuzzy_0': True,
  'incalls_fuzzy_0': True,
  'peakvce_fuzzy_0': True,
  'opeakvce_fuzzy_1': True,
  'dropblk_fuzzy_0': True,
  'callfwdv_fuzzy_0': True,
  'callwait_fuzzy_0': True,
  'months_fuzzy_1': True,
  'uniqsubs_fuzzy_1': True,
  'actvsubs_fuzzy_0': True,
  'phones_fuzzy_0': True,
  'models_fuzzy_0': True,
  'eqpdays_fuzzy_1': True,
  'age1_fuzzy_1': True,
  'age2_fuzzy_1': True,
  'children_fuzzy_2': True,
  'credita_fuzzy_0': True,
  'creditaa_fuzzy_0': True,
  'prizmrur_fuzzy_0': True,
  'prizmub_fuzzy_0': True,
  'prizmtwn_fuzzy_2': True,
  'refurb_fuzzy_0': True,
  'webcap_fuzzy

### High Utility fuzzy Churn Patterns

In [31]:
import time

def extract_base_name(fuzzy_name):
    if fuzzy_name.startswith("Status_"):
        return "Status"
    elif fuzzy_name.startswith("Age Group_"):
        return "Age Group"
    elif fuzzy_name.startswith("Tariff Plan_"):
        return "Tariff Plan"
    elif "fuzzy" in fuzzy_name:
        return fuzzy_name.split("_", 1)[0]
    else:
        return fuzzy_name

def calculate_utility(itemset, transactions, profit_table):
    total_utility = 0
    for transaction in transactions.values():
        if set(itemset).issubset(set(transaction.keys())):
            for item, utility in transaction.items():
                if item in itemset:
                    base_name = extract_base_name(item)
                    total_utility += utility * profit_table.get(base_name, 0)
    return total_utility

def find_top_k_high_utility_itemsets(transactions, profit_table, k):
    itemsets = set()
    for transaction in transactions.values():
        itemsets.update(transaction.keys())

    high_utility_itemsets = {}
    for item in itemsets:
        utility = calculate_utility([item], transactions, profit_table)
        if utility > 0:
            high_utility_itemsets[(item,)] = utility

    P_itemsets = {}
    while True:
        temp_itemsets = {}
        for itemset, utility in high_utility_itemsets.items():
            for item in itemsets:
                if item not in itemset:
                    new_itemset = tuple(sorted(list(itemset) + [item]))
                    new_utility = calculate_utility(new_itemset, transactions, profit_table)
                    if new_utility > 0:
                        temp_itemsets[new_itemset] = new_utility
                        P_itemsets[new_itemset] = new_utility

        if not temp_itemsets:
            break

        sorted_itemsets = sorted(temp_itemsets.items(), key=lambda x: x[1], reverse=True)
        top_k_itemsets = sorted_itemsets[:k]
        high_utility_itemsets = {itemset: utility for itemset, utility in top_k_itemsets}

    return high_utility_itemsets, P_itemsets

def find_top_k_high_utility_itemsets(transactions, profit_table, k, desired_length):
    """
    Tìm ra K tập mục có độ hữu ích cao nhất với độ dài tối đa được chỉ định.
    """
    itemsets = set()
    for transaction in transactions.values():
        # Thêm từng mục trong giao dịch vào tập hợp itemsets
        itemsets.update(transaction.keys())

    # Lưu trữ các tập mục có độ hữu ích cao và tổng giá trị của chúng
    high_utility_itemsets = {}
    P_itemsets = {}

    for item in itemsets:
        utility = calculate_utility([item], transactions, profit_table)
        if utility > 0:
            high_utility_itemsets[(item,)] = utility

    while True:
        temp_itemsets = {}
        for itemset, utility in high_utility_itemsets.items():
            for item in itemsets:
                if item not in itemset:
                    # Tạo tập mục mới bằng cách kết hợp
                    new_itemset = tuple(sorted(list(itemset) + [item]))
                    new_utility = calculate_utility(new_itemset, transactions, profit_table)

                    # Kiểm tra điều kiện độ dài và độ hữu ích
                    if new_utility > 0 and len(new_itemset) <= desired_length:
                        temp_itemsets[new_itemset] = new_utility
                        P_itemsets[new_itemset] = new_utility

        if not temp_itemsets:
            break

        # Lấy top K tập mục có độ hữu ích cao nhất
        sorted_itemsets = sorted(temp_itemsets.items(), key=lambda x: x[1], reverse=True)
        top_k_itemsets = sorted_itemsets[:k]
        high_utility_itemsets = {itemset: utility for itemset, utility in top_k_itemsets}

    return high_utility_itemsets, P_itemsets



In [32]:
result,P_itemsets= find_top_k_high_utility_itemsets(filtered_transactions, profit_table, 3,2)


sorted_data = sorted(P_itemsets.items(), key=lambda x: x[1], reverse=True)


top_10 = sorted_data[:10]
top_10 = {itemset: value for itemset, value in top_10}

print("Top 10 High Utility Itemsets:")
for itemset, utility in top_10.items():
    print(itemset, "-> Utility:", round(utility,0))

Top 10 High Utility Itemsets:
('changem_fuzzy_1', 'eqpdays_fuzzy_1') -> Utility: 1902.0
('eqpdays_fuzzy_1', 'months_fuzzy_1') -> Utility: 1609.0
('creditaa_fuzzy_0', 'eqpdays_fuzzy_1') -> Utility: 1506.0
('eqpdays_fuzzy_1', 'mou_fuzzy_1') -> Utility: 1493.0
('changer_fuzzy_1', 'eqpdays_fuzzy_1') -> Utility: 1457.0
('eqpdays_fuzzy_1', 'recchrge_fuzzy_1') -> Utility: 1428.0
('eqpdays_fuzzy_1', 'webcap_fuzzy_1') -> Utility: 1350.0
('credita_fuzzy_0', 'eqpdays_fuzzy_1') -> Utility: 1349.0
('eqpdays_fuzzy_1', 'retcalls_fuzzy_0') -> Utility: 1346.0
('changem_fuzzy_1', 'months_fuzzy_1') -> Utility: 1324.0


In [33]:

fuzzy_df = pd.DataFrame()
for feature in Xtofuzzy_Tri.columns:
    min_val = Xtofuzzy_Tri[feature].min()
    median_val = Xtofuzzy_Tri[feature].quantile(0.5)
    max_val = Xtofuzzy_Tri[feature].max()

    fuzzy_df[f'{feature}_fuzzy'] = Xtofuzzy_Tri[feature].apply(lambda x: fuzzy_classification(x, min_val, median_val, max_val))


df_fuzzy = fuzzy_df

fuzzy_columns = [col for col in df_fuzzy.columns]

df_fuzzy = pd.get_dummies(df_fuzzy, columns=fuzzy_columns)
df_fuzzy =df_fuzzy.astype(int)
df_fuzzy

,revenue_fuzzy_0,revenue_fuzzy_1,revenue_fuzzy_2,mou_fuzzy_0,mou_fuzzy_1,mou_fuzzy_2,recchrge_fuzzy_0,recchrge_fuzzy_1,recchrge_fuzzy_2,directas_fuzzy_0,...,income_fuzzy_2,mcycle_fuzzy_0,mcycle_fuzzy_2,setprcm_fuzzy_0,setprcm_fuzzy_1,setprc_fuzzy_0,setprc_fuzzy_1,setprc_fuzzy_2,retcall_fuzzy_0,retcall_fuzzy_2
54569,0,1,0,0,1,0,0,1,0,1,...,0,0,1,0,1,1,0,0,1,0
52439,0,1,0,0,1,0,0,1,0,1,...,0,1,0,1,0,0,1,0,1,0
63639,0,1,0,1,0,0,0,1,0,1,...,0,1,0,0,1,1,0,0,1,0
57383,0,1,0,0,1,0,0,1,0,0,...,0,1,0,1,0,0,1,0,1,0
70085,0,1,0,0,1,0,0,1,0,1,...,0,1,0,0,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55990,0,1,0,0,1,0,0,1,0,1,...,0,1,0,0,1,1,0,0,1,0
53067,0,1,0,1,0,0,0,1,0,1,...,1,1,0,0,1,1,0,0,1,0
53682,0,1,0,0,1,0,0,1,0,1,...,0,1,0,1,0,0,1,0,1,0
51952,0,1,0,0,1,0,0,1,0,1,...,1,1,0,1,0,0,1,0,1,0


In [34]:
Xy_merge = pd.merge(X_temp_test, df_fuzzy, left_index=True, right_index=True, how='left')
Xy_merge['fuzzyp1'] = Xy_merge['changem_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp2'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['months_fuzzy_1']
Xy_merge['fuzzyp3'] = Xy_merge['creditaa_fuzzy_0'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp4'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['mou_fuzzy_1']
Xy_merge['fuzzyp5'] = Xy_merge['actvsubs_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp6'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['recchrge_fuzzy_1']
Xy_merge['fuzzyp7'] = Xy_merge['changer_fuzzy_1'] * Xy_merge['eqpdays_fuzzy_1']
Xy_merge['fuzzyp8'] = Xy_merge['changem_fuzzy_1'] * Xy_merge['months_fuzzy_1']
Xy_merge['fuzzyp9'] = Xy_merge['eqpdays_fuzzy_1'] * Xy_merge['webcap_fuzzy_1']
Xy_merge['fuzzyp10'] = Xy_merge['credita_fuzzy_0'] * Xy_merge['eqpdays_fuzzy_1']

print(Xy_merge.columns.tolist())


['revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer', 'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls', 'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs', 'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur', 'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft', 'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord', 'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly', 'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall', 'churn', 'revenue_fuzzy_0', 'revenue_fuzzy_1', 'revenue_fuzzy_2', 'mou_fuzzy_0', 'mou_fuzzy_1', 'mou_fuzzy_2', 'recchrge_fuzzy_0', 'recchrge_fuzzy_1', 'recchrge_fuzzy_2', 'directas_fuzzy_0', 'directas_fuzzy_1', 'directas_fuzzy_2', 'overage_fuzzy_0', 'overage_fuzzy_1', 'overage_fuzzy_2'

In [35]:
Xy_merge = Xy_merge.fillna(0)

In [36]:
columns_to_keep = [
    'revenue', 'mou', 'recchrge', 'directas', 'overage', 'roam', 'changem', 'changer',
    'dropvce', 'blckvce', 'unansvce', 'custcare', 'threeway', 'mourec', 'outcalls', 'incalls',
    'peakvce', 'opeakvce', 'dropblk', 'callfwdv', 'callwait', 'months', 'uniqsubs', 'actvsubs',
    'phones', 'models', 'eqpdays', 'age1', 'age2', 'children', 'credita', 'creditaa', 'prizmrur',
    'prizmub', 'prizmtwn', 'refurb', 'webcap', 'truck', 'rv', 'occprof', 'occcler', 'occcrft',
    'occstud', 'occhmkr', 'occret', 'occself', 'ownrent', 'marryun', 'marryyes', 'mailord',
    'mailres', 'mailflag', 'travel', 'pcown', 'creditcd', 'retcalls', 'retaccpt', 'newcelly',
    'newcelln', 'refer', 'incmiss', 'income', 'mcycle', 'setprcm', 'setprc', 'retcall',
    'fuzzyp1', 'fuzzyp2', 'fuzzyp3', 'fuzzyp4', 'fuzzyp5', 'fuzzyp6', 'fuzzyp7', 'fuzzyp8',
    'fuzzyp9', 'fuzzyp10'
]
X_test_2 = Xy_merge[columns_to_keep]
y_test_2 = Xy_merge['churn']


# Final Model

In [37]:
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix, classification_report)
import time
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
import optuna


overall_start_time = time.time()


smote = SMOTE(random_state=42)
X_resampled_2, y_resampled_2 = smote.fit_resample(X_train_2, y_train_2)

def objective(trial):
    """Hàm mục tiêu của Optuna để tìm siêu tham số tối ưu."""
    max_depth = trial.suggest_int('max_depth', 2, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 500)

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        verbosity=0,
        use_label_encoder=False
    )

    model.fit(X_resampled_2, y_resampled_2)
    y_prob_2 = model.predict_proba(X_test_2)[:, 1]

    return roc_auc_score(y_test_2, y_prob_2)

start_time = time.time()

# Thiết lập Optuna để tìm kiếm siêu tham số
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)


end_time = time.time()

total_time = end_time - start_time
print(f"Thời gian tìm kiếm siêu tham số: {total_time:.2f} giây")

best_params = study.best_params
print("Siêu tham số tốt nhất:", best_params)

best_model = xgb.XGBClassifier(
    objective='binary:logistic',
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    n_estimators=best_params['n_estimators'],
    verbosity=0,
    use_label_encoder=False
)
best_model.fit(X_train_2, y_train_2)

y_pred = best_model.predict(X_test_2)
y_prob = best_model.predict_proba(X_test_2)[:, 1]

auc_score = roc_auc_score(y_test_2, y_prob)
print("AUC:", auc_score)
print("Accuracy:", accuracy_score(y_test_2, y_pred))
print("Precision:", precision_score(y_test_2, y_pred))
print("Recall:", recall_score(y_test_2, y_pred))
print("F1 Score:", f1_score(y_test_2, y_pred))
print("Matthews Correlation Coefficient:", matthews_corrcoef(y_test_2, y_pred))
print("\nMa trận nhầm lẫn:")
print(confusion_matrix(y_test_2, y_pred))
print("\nBáo cáo phân loại:")
print(classification_report(y_test_2, y_pred))

# Ghi lại thời gian kết thúc toàn bộ quá trình
overall_end_time = time.time()

# Tính tổng thời gian thực thi
overall_total_time = overall_end_time - overall_start_time
print(f"Tổng thời gian thực thi: {overall_total_time:.2f} giây")


[I 2025-12-15 23:07:17,618] A new study created in memory with name: no-name-8a41e993-1fa1-4dd7-9f75-786e0bc1e903
[I 2025-12-15 23:07:17,973] Trial 0 finished with value: 0.9971058501322028 and parameters: {'max_depth': 2, 'learning_rate': 0.061839898830968465, 'n_estimators': 317}. Best is trial 0 with value: 0.9971058501322028.
[I 2025-12-15 23:07:18,585] Trial 1 finished with value: 0.9971138333461816 and parameters: {'max_depth': 7, 'learning_rate': 0.04187150526326223, 'n_estimators': 285}. Best is trial 1 with value: 0.9971138333461816.
[I 2025-12-15 23:07:19,295] Trial 2 finished with value: 0.9970482669178584 and parameters: {'max_depth': 4, 'learning_rate': 0.2628971215459385, 'n_estimators': 447}. Best is trial 1 with value: 0.9971138333461816.
[I 2025-12-15 23:07:19,658] Trial 3 finished with value: 0.997554623523432 and parameters: {'max_depth': 2, 'learning_rate': 0.1831270164970193, 'n_estimators': 315}. Best is trial 3 with value: 0.997554623523432.
[I 2025-12-15 23:07:1

Thời gian tìm kiếm siêu tham số: 128.47 giây
Siêu tham số tốt nhất: {'max_depth': 4, 'learning_rate': 0.04089466384184448, 'n_estimators': 425}
AUC: 0.9977657154390104
Accuracy: 0.981604386091473
Precision: 0.9931362196409715
Recall: 0.9426208970182911
F1 Score: 0.9672194369456228
Matthews Correlation Coefficient: 0.9550745911205134

Ma trận nhầm lẫn:
[[9845   26]
 [ 229 3762]]

Báo cáo phân loại:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      9871
           1       0.99      0.94      0.97      3991

    accuracy                           0.98     13862
   macro avg       0.99      0.97      0.98     13862
weighted avg       0.98      0.98      0.98     13862

Tổng thời gian thực thi: 129.17 giây
